# source :
https://github.com/allenai/olmocr/tree/main/olmocr/bench

## Setup

In [ ]:
#setup
!sudo apt-get update
!sudo apt-get install poppler-utils ttf-mscorefonts-installer msttcorefonts fonts-crosextra-caladea fonts-crosextra-carlito gsfonts lcdf-typetools

!pip install olmocr[gpu]  --extra-index-url https://download.pytorch.org/whl/cu128

# Option : Install flash infer for faster inference on GPU
# pip install https://download.pytorch.org/whl/cu128/flashinfer/flashinfer_python-0.2.5%2Bcu128torch2.7-cp38-abi3-linux_x86_64.whl

!playwright install chromium

## Download dataset

In [3]:
#dataset for olmOCR-bench
!hf download --repo-type dataset --resume-download allenai/olmOCR-bench --local-dir ./olmOCR-bench

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 1417 files:   0%|                             | 0/1417 [00:00<?, ?it/s]Downloading 'bench_data/arxiv_math.jsonl' to 'olmOCR-bench/.cache/huggingface/download/bench_data/fGbZRoezupRHykPZ8TgvdwfBMX0=.01442bacc58d81be7f65890753ef6ee37c1d18cd.incomplete'

arxiv_math.jsonl: 0.00B [00:00, ?B/s]Downloading '.DS_Store' to 'olmOCR-bench/.cache/huggingface/download/aHCrtOA_94YSUtnsyba37B_tPq0=.49e589b2f060915be758eec1d071f14e93da1203.incomplete'


.gitignore: 100%|██████████████████████████████| 149/149 [00:00<00:00, 1.40MB/s]
Download complete. Moving file to olmOCR-bench/.gitignore


.DS_Store: 100%|

# Step 1 : convert pdf to markdown
* --dir to directory your file for benchmark
* You can change the pipeline to another pipeline, for example:
```
    python -m bench.convert {other_pipeline} --dir {./your_data}
```
* pdf must in pdfs folder
### stucture folder :
     bench_data/
     ├── olmocr_pipeline/   # outputfile .md
     |   ├── math/  
     │   ├── table/
     |   └── ...
     └── pdfs/      # .pdf original files
         ├── math/  
         ├── table/
         └── ...  


In [1]:
!pwd

/teamspace/studios/this_studio/olmocr/olmocr


In [ ]:
!python -m bench.convert olmocr_pipeline --dir /teamspace/studios/this_studio/olmOCR-bench/testfile --parallel 4

`olmocr_pipeline` method requested. Starting VLLM server...
Waiting for VLLM server to be ready...
Processing finished. Shutting down VLLM server...
^C
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/site-packages/httpx/_transports/default.py", line 394, in handle_async_request
    resp = await self._pool.handle_async_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/site-packages/httpcore/_async/connection_pool.py", line 256, in handle_async_request
    raise exc from None
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/site-packages/httpcore/_async/connection_pool.py", line 236, in handle_async_request
    response = await connection.handle_async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^

# Step 2 : run the benchmark
* --dir specifies the directory where the benchmark data is stored
* output filename format :
{base_name}_pg{page}_repeat{repeat}.md
### stucture folder :
     bench_data/
     ├── olmocr_pipeline/   # file .md after running the convert command
     |   ├── math/  
     │   ├── table/
     |   └── ...
     ├── pdfs/      # file .pdf original files
     │   ├── math/  
     │   ├── table/
     |   └── ...    # other data
     ├── math.jsonl/  # metadata file for the benchmark
     ├── table.jsonl/ # metadata file for the benchmark
     └── ...josnl     # other metadata files for the benchmark

### structure Jsonl :
          {"pdf": "arxiv_math/2503.05177_pg10.pdf", # filename
          "url": "https://arxiv.org/pdf/2503.05177", #url file NOT IMPORTANT
          "page": 1, #page in file
          "id": "2503.05177_pg10_math_002", # id for refer
          "type": "math", # type of doc e.g. math , table , absent
          "max_diffs": 0,
          "checked": null,
          "math": "P(X_1 = X_2 = \\dots =X_{n_0} = 1) > 0"} # ข้อมูลที่ต้องการเช็ค

In [2]:
!python -m bench.benchmark --dir /teamspace/studios/this_studio/olmOCR-bench/testfile --skip_baseline

Loading tests: 100%|██████████████████████████████| 2/2 [00:02<00:00,  1.49s/it]

Running tests for each candidate:

Evaluating candidate: olmocr_pipeline
Evaluating tests for olmocr_pipeline: 100%|█████| 2/2 [00:06<00:00,  3.50s/test]

Candidate: olmocr_pipeline
  [FAIL] Test 2503.04415_pg2_math_001 on arxiv_math/2503.04415_pg2 page 1 average pass ratio: 0.000 (0/1 repeats passed). Ex: No match found for N = \left\lfloor \frac{1}{\gamma} \right\rfloor anywhere in content
  Average Score: 50.0% (95% CI: [0.0%, 100.0%]) over 2 tests.

Final Summary with 95% Confidence Intervals:
olmocr_pipeline      : Average Score: 50.0% ± 50.0% (average of per-JSONL scores)
    math    : 50.0% average pass rate over 2 tests

    Results by JSONL file:
        arxiv_math.jsonl              : 50.0% (1/2 tests)

